## 6_urban_rural_segregation
### Categorizes buildings based on their location into a fitting urbanization status, uses the pre-processed polygons from previous notebook - 5_rural_urban_json_segregation
### This notebook needs to be executed twice together with 5_rural_urban_json_segregation to categorize building on a "overview" level, i.e., urban-suburban-rural categories and second time to categorize buildings into finer grainde categories using the "detailed" option SEGREGATION_TYPE

### To remove connection to IBM COS save the boundary of the area of interest on a local computer and 
 - Ignore the 4rd cell starting with "cos_client = ibm_boto3.client"
 - In the 6th do not run the downloading script, but make sure that the output of the 5_rural_urban_json_segregation script is present in the location
 - In the last cell remove the re-upload of intermediate results (which are the last lines of the cell)

### Initial configuration
#### To start working with this particular notebook, you need to provide necessary credential and settings
#### Below is an template of configuration, which is necessary prepare aside of this notebook and copy & paste all content in triple quotes to the next cell's input field
    """
    {
    "COUNTRY_NAME": "Kenya",
    "SEGREGATION_STYLE": "overview",
    "VIDA_COUNTRIES_BUILDINGS": "vida-countries-buildings",

    "COS_ENDPOINT_URL": "s3.private.eu-de.cloud-object-storage.appdomain.cloud",
    "COS_AUTH_ENDPOINT_URL": "https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "COUNTRY_TABLE": "FEATURES_DB_VIDA_EXTENDED",
    "SMOD_BUCKET": "xxx",
    }
    """


In [53]:
# Read notebook configuration
import getpass
import json

#config_str = getpass.getpass('Enter your prepared config: ')
config_str = '''

{

    "COUNTRY_NAME": "India",
    "VIDA_COUNTRIES_BUILDINGS": "vida-countries-buildings",
    "COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
    "COS_AUTH_ENDPOINT_URL": "https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "t_tpW55SqTHzGaj6ZsTTiGxkBDUsTQ7RjWO17qKoh_B8",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "COUNTRY_TABLE": "FEATURES_DB_VIDA_EXTENDED",
    "OUTPUT_BUCKET": "onboarding-bucket-s"

}

'''
config = json.loads(config_str)

In [54]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd
import jaydebeapi as jdbc
import jpype
import ibm_boto3
import gc
import io
import os
import shapely
import numpy as np
import rasterio as rio
from botocore.client import Config
from rasterio.plot import show
from tqdm import tqdm
from rasterio.mask import mask
import matplotlib.pyplot as plt
import rioxarray
from skimage import measure as M
import json

In [55]:

min_latitude = 0
max_latitude = 0
min_longitude = 0
max_longitude = 0

# Overview names
segregation_overview_names = {
    'URBAN': 'Urban',
    'SUBURBAN': 'Suburban',
    'RURAL': 'Rural'
}

# Detailed names
segregation_detailed_names = {
    'URBAN_CENTER': 'Urban Center',
    'DENSE_URBAN': 'Dense Urban Cluster',
    'SEMI_DENSE_URBAN': 'Semi-dense Urban Cluster',
    'SUBURBAN_PERI_URBAN': 'Suburban or Peri-urban',
    'RURAL_CLUSTER': 'Rural Cluster',
    'LOW_DENSITY_RURAL': 'Low Density Rural'
}


In [56]:
#load db2jcc4.jar and polygons json
cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])

response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

try:
    for obj in response['Contents']:
        name = obj['Key']
        streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
        print("Copying to localStorage :  " + name)
        with io.FileIO(name, 'w') as file:
            for i in io.BytesIO(streaming_body_1.read()):
                file.write(i)
    
    from utils import *
    print('External utils succesfully imported')
    
except Exception as e:
    print('Error occured: ', e)

Copying to localStorage :  S2_MGRS_tile_names.json
Copying to localStorage :  db2jcc4.jar
Copying to localStorage :  filtering_grid_generator.py
Copying to localStorage :  india_state.geojson
Copying to localStorage :  regions_S2_ids.pkl
Copying to localStorage :  utils.py
External utils succesfully imported


In [57]:
smod_overview_json_filenames = ['Churu_overview.json']
smod_detailed_json_filenames = ['Churu_detailed.json']

vida_datasets = ['Churu_buildings.parquet']


In [58]:
# Fetch the geoJSON file containing polygons and process each polygon one by one
for smod_json in smod_overview_json_filenames + smod_detailed_json_filenames:
    
    streaming_body = cos_client.get_object(Bucket=config["OUTPUT_BUCKET"], Key=smod_json)['Body']
    print("Downloading to local storage :  " + smod_json)
    with io.FileIO(smod_json, 'w') as file:
        for i in io.BytesIO(streaming_body.read()):
            file.write(i)

# Fetch the geoJSON file containing polygons and process each polygon one by one
for vida_buildings_parquet in vida_datasets:
    
    #streaming_body = cos_client.get_object(Bucket=config["VIDA_COUNTRIES_BUILDINGS"], Key=vida_buildings_parquet)['Body']
    streaming_body = cos_client.get_object(Bucket=config["OUTPUT_BUCKET"], Key=vida_buildings_parquet)['Body']
    print("Downloading to local storage :  " + vida_buildings_parquet)
    with io.FileIO(vida_buildings_parquet, 'w') as file:
        for i in io.BytesIO(streaming_body.read()):
            file.write(i)



In [59]:
smod_overview_json_filenames.sort(), smod_detailed_json_filenames.sort(), vida_datasets.sort()
smod_overview_json_filenames, smod_detailed_json_filenames, vida_datasets

(['Churu_overview.json'], ['Churu_detailed.json'], ['Churu_buildings.parquet'])

In [60]:
for smod_json_overview, smod_json_detailed, parquet_filename in zip(smod_overview_json_filenames, smod_detailed_json_filenames, vida_datasets):
#Open Kenya_segregated polygons

     # === OVERVIEW PROCESSING ===
    with open(smod_json_overview) as s_f:
        geojson_overview = json.load(s_f)

    print('Processing: ', smod_json_overview, smod_json_detailed, parquet_filename)

    buildings_df = gpd.read_parquet(parquet_filename)
    init_len = len(buildings_df)
    dfs = []
    
    for fidx, feature in enumerate(tqdm(geojson_overview['features'], total=len(geojson_overview['features']), desc='Processing overview level')):
        polygon_coordinates = feature['geometry']['coordinates'][0]
        polygon = shapely.Polygon(polygon_coordinates)
        min_lon, min_lat, max_lon, max_lat = polygon.bounds
        
        buildings_bbox = buildings_df[
                (buildings_df.longitude >= min_lon) &
                (buildings_df.longitude <= max_lon) &
                (buildings_df.latitude >= min_lat) &
                (buildings_df.latitude <= max_lat)
        ].copy()
        
        buildings_bbox['buildings_in_polygon'] = [polygon.contains(shapely.Point(row.longitude, row.latitude)) for row in buildings_bbox.itertuples()]

        buildings_in_polygon = buildings_bbox[buildings_bbox.buildings_in_polygon == True]        
        
        buildings_in_polygon = buildings_in_polygon.drop(columns=['buildings_in_polygon'])
        buildings_in_polygon['urban_split'] = segregation_overview_names [feature['properties']['seg_type']]
        
        dfs.append(buildings_in_polygon)
     
    try: 
        segregated_df = pd.concat(dfs)
        
        remaining_buildings = buildings_df[~buildings_df['id'].isin(segregated_df['id'])]
        remaining_buildings['urban_split'] = 'Rural'
        
        main_df = pd.concat([segregated_df, remaining_buildings])
        
        print(f'init len {init_len} out len {len(main_df)}')
    except Exception as e:
        print(e)
        
    #################################################################################################################
    print('removing duplicates')
    main_df = main_df.drop_duplicates(subset='id')
    print(f'in len {init_len} out len {len(main_df)}')

    main_df.index = [i for i in range(len(main_df))]

     # === DETAILED PROCESSING ===
    with open(smod_json_detailed) as s_f:
        geojson_detailed = json.load(s_f)
        
    dfs = []
        
    for fidx, feature in enumerate(tqdm(geojson_detailed['features'], total=len(geojson_detailed['features']), desc='Processing detailed level')):
        
        polygon_coordinates = feature['geometry']['coordinates'][0]
        polygon = shapely.Polygon(polygon_coordinates)
        min_lon, min_lat, max_lon, max_lat = polygon.bounds
        
        buildings_bbox = main_df[
                (main_df.longitude >= min_lon) &
                (main_df.longitude <= max_lon) &
                (main_df.latitude >= min_lat) &
                (main_df.latitude <= max_lat)
        ].copy()
        
        buildings_bbox['buildings_in_polygon'] = [polygon.contains(shapely.Point(row.longitude, row.latitude)) for row in buildings_bbox.itertuples()]

        buildings_in_polygon = buildings_bbox[buildings_bbox.buildings_in_polygon == True]        
        
        buildings_in_polygon = buildings_in_polygon.drop(columns=['buildings_in_polygon'])
        buildings_in_polygon['ghsl_smod'] = segregation_detailed_names[feature['properties']['seg_type']]
        
        dfs.append(buildings_in_polygon)
     
    try: 
        segregated_df = pd.concat(dfs)
        
        remaining_buildings = main_df[~main_df['id'].isin(segregated_df['id'])]
        remaining_buildings['ghsl_smod'] = 'Low Density Rural'
        
        main_df = pd.concat([segregated_df, remaining_buildings])
        
        main_df = main_df.drop_duplicates(subset='id')
        
        print(f'in len {init_len} out len {len(main_df)}')
    except Exception as e:
        print(e)
        
    try:
        
        filename = parquet_filename.replace('.parquet', '_GHSL_SMOD.parquet')
        main_df.to_parquet(filename)
        
        res=cos_client.upload_file(Filename=filename, Bucket=config["VIDA_COUNTRIES_BUILDINGS"],Key=filename)
        print(f'File uploaded to the COS {filename}')
    except Exception as e:
        print(e)

Processing:  Churu_overview.json Churu_detailed.json Churu_buildings.parquet


Processing overview level: 100%|██████████| 164/164 [00:09<00:00, 17.77it/s]
/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


init len 1481134 out len 1481134
removing duplicates
in len 1481134 out len 1481134


Processing detailed level: 100%|██████████| 1416/1416 [00:36<00:00, 38.29it/s]
/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


in len 1481134 out len 1481134
File uploaded to the COS Churu_buildings_GHSL_SMOD.parquet
